# Sentiment Classification


## Loading the dataset

In [1]:
from keras.datasets import imdb

vocab_size = 10000 #vocab size

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size) # vocab_size is no.of words to consider from the dataset, ordering based on frequency.

Using TensorFlow backend.


17465344/17464789 [==============================] - 1s 0us/step


In [2]:
x_test[1]

[1,
 14,
 22,
 3443,
 6,
 176,
 7,
 5063,
 88,
 12,
 2679,
 23,
 1310,
 5,
 109,
 943,
 4,
 114,
 9,
 55,
 606,
 5,
 111,
 7,
 4,
 139,
 193,
 273,
 23,
 4,
 172,
 270,
 11,
 7216,
 2,
 4,
 8463,
 2801,
 109,
 1603,
 21,
 4,
 22,
 3861,
 8,
 6,
 1193,
 1330,
 10,
 10,
 4,
 105,
 987,
 35,
 841,
 2,
 19,
 861,
 1074,
 5,
 1987,
 2,
 45,
 55,
 221,
 15,
 670,
 5304,
 526,
 14,
 1069,
 4,
 405,
 5,
 2438,
 7,
 27,
 85,
 108,
 131,
 4,
 5045,
 5304,
 3884,
 405,
 9,
 3523,
 133,
 5,
 50,
 13,
 104,
 51,
 66,
 166,
 14,
 22,
 157,
 9,
 4,
 530,
 239,
 34,
 8463,
 2801,
 45,
 407,
 31,
 7,
 41,
 3778,
 105,
 21,
 59,
 299,
 12,
 38,
 950,
 5,
 4521,
 15,
 45,
 629,
 488,
 2733,
 127,
 6,
 52,
 292,
 17,
 4,
 6936,
 185,
 132,
 1988,
 5304,
 1799,
 488,
 2693,
 47,
 6,
 392,
 173,
 4,
 2,
 4378,
 270,
 2352,
 4,
 1500,
 7,
 4,
 65,
 55,
 73,
 11,
 346,
 14,
 20,
 9,
 6,
 976,
 2078,
 7,
 5293,
 861,
 2,
 5,
 4182,
 30,
 3127,
 2,
 56,
 4,
 841,
 5,
 990,
 692,
 8,
 4,
 1669,
 398,
 229,
 10,


In [0]:
from keras.preprocessing.sequence import pad_sequences
vocab_size = 10000 #vocab size
maxlen = 300  #number of word used from each review

## Train test split

In [0]:
#load dataset as a list of ints
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)
#make all sequences of the same length
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test =  pad_sequences(x_test, maxlen=maxlen)

## Build Keras Embedding Layer Model
We can think of the Embedding layer as a dicionary that maps a index assigned to a word to a word vector. This layer is very flexible and can be used in a few ways:

* The embedding layer can be used at the start of a larger deep learning model. 
* Also we could load pre-train word embeddings into the embedding layer when we create our model.
* Use the embedding layer to train our own word2vec models.

The keras embedding layer doesn't require us to onehot encode our words, instead we have to give each word a unqiue intger number as an id. For the imdb dataset we've loaded this has already been done, but if this wasn't the case we could use sklearn [LabelEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2, 
    random_state=42
)

In [0]:
#Initialize model
import tensorflow as tf
tf.keras.backend.clear_session()
model = tf.keras.Sequential()

In [7]:
model.add(tf.keras.layers.Embedding(vocab_size + 1, #Vocablury size
                                    50, #Embedding size
                                    input_length=maxlen) #Number of words in each review
          )

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
If using Keras pass *_constraint arguments to layers.


## Retrive the output of each layer in keras for a given single test sample from the trained model you built

In [0]:
model.add(tf.keras.layers.LSTM(256, #RNN State - size of cell state and hidden state
                               dropout=0.2, #Dropout before feeding the data to LSTM layer
                               recurrent_dropout=0.2)) #Dropout applied to the output of LSTM layer

In [9]:
model.output

<tf.Tensor 'lstm/strided_slice_7:0' shape=(?, 256) dtype=float32>

In [0]:
model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

In [11]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


In [12]:
model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding (Embedding)        (None, 300, 50)           500050    
_________________________________________________________________
lstm (LSTM)                  (None, 256)               314368    
_________________________________________________________________
dense (Dense)                (None, 1)                 257       
Total params: 814,675
Trainable params: 814,675
Non-trainable params: 0
_________________________________________________________________


In [13]:
model.fit(X_train,y_train,
          epochs=10,
          batch_size=32,          
          validation_data=(X_val, y_val))

Train on 20000 samples, validate on 5000 samples
Epoch 1/10
20000/20000 [==============================] - 394s 20ms/sample - loss: 0.5274 - acc: 0.7423 - val_loss: 0.5008 - val_acc: 0.7544
Epoch 2/10
20000/20000 [==============================] - 391s 20ms/sample - loss: 0.3708 - acc: 0.8442 - val_loss: 0.4340 - val_acc: 0.8116
Epoch 3/10
20000/20000 [==============================] - 389s 19ms/sample - loss: 0.3054 - acc: 0.8769 - val_loss: 0.3835 - val_acc: 0.8348
Epoch 4/10
20000/20000 [==============================] - 386s 19ms/sample - loss: 0.2629 - acc: 0.8955 - val_loss: 0.3953 - val_acc: 0.8318
Epoch 5/10
20000/20000 [==============================] - 385s 19ms/sample - loss: 0.3238 - acc: 0.8608 - val_loss: 0.3642 - val_acc: 0.8540
Epoch 6/10
20000/20000 [==============================] - 388s 19ms/sample - loss: 0.2065 - acc: 0.9216 - val_loss: 0.3442 - val_acc: 0.8752
Epoch 7/10
20000/20000 [==============================] - 387s 19ms/sample - loss: 0.1666 - acc: 0.9373 -